# 🚀 Eksperimen Alternatif 1: Instance-Based Filtering & Transfer Learning
## (Burak Filter & TrAdaBoost untuk Mitigasi Domain Shift)

- **Kelompok Peneliti**: Kelompok A01 — Rekayasa Perangkat Lunak A (Magister Teknik Informatika ITS)
- **Hardware Akselerasi**: Laptop GPU NVIDIA GeForce RTX 4050 (6GB GDDR6 VRAM, CUDA 13.4, Driver 617.42)
- **Dataset**: FlakeFlagger (22.236 test cases dari 24 repositori Java)
- **Paper Rujukan**: 
  1. Afeltra et al. (IEEE Access 2024)
  2. Turhan et al. (IEEE TSE 2009 — *Burak Filter*)
  3. Dai et al. (ICML 2007 — *TrAdaBoost*)
  4. Alshammari et al. (ICSE 2021 — *FlakeFlagger*)

### 1. Verifikasi Lingkungan GPU NVIDIA RTX 4050 & CUDA 13

In [ ]:
import sys
from pathlib import Path

# Tambahkan path src ke sys.path
src_path = Path("../src").resolve()
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

from check_gpu import check_nvidia_smi, check_xgboost_gpu
check_nvidia_smi()
check_xgboost_gpu()

### 2. Pemuatan & Pemeriksaan Dataset FlakeFlagger (C1: Row-Level Drop)

In [ ]:
from data_loader import load_dataset

df, features, projects = load_dataset("C1")
print(f"Total baris: {len(df):,}")
print(f"Total fitur prediktif: {len(features)}")
print(f"Jumlah proyek: {len(projects)}")
print(f"Total flaky tests: {df['flaky'].sum()} ({df['flaky'].mean()*100:.2f}%)")

### 3. Eksekusi LOPO-CV untuk 4 Konfigurasi Model Transfer Learning

In [ ]:
from transfer_lopo_runner import run_transfer_lopo_experiment

df_folds, df_summary = run_transfer_lopo_experiment(
    config_path="../configs/transfer_config.json",
    output_dir="../results"
)

display(df_summary)

### 4. Uji Signifikansi Statistik (Wilcoxon Signed-Rank Test & Cliff's Delta)

In [ ]:
from statistical_tests import run_wilcoxon_analysis

df_wilcoxon = run_wilcoxon_analysis(
    folds_csv_path="../results/transfer_learning_folds.csv",
    output_file="../results/wilcoxon_test_results.csv"
)

display(df_wilcoxon[df_wilcoxon["Metric"].isin(["F1", "PR_AUC"])])

### 5. Pembuatan dan Tampilan Visualisasi Publikasi Ilmiah

In [ ]:
from visualizer import generate_all_plots
from IPython.display import Image, display

generate_all_plots(
    folds_csv_path="../results/transfer_learning_folds.csv",
    summary_csv_path="../results/transfer_learning_summary.csv",
    plots_dir="../results/plots"
)

display(Image(filename="../results/plots/boxplot_comparison_f1_prauc.png"))
display(Image(filename="../results/plots/barchart_metrics_comparison.png"))
display(Image(filename="../results/plots/gpu_runtime_efficiency.png"))
display(Image(filename="../results/plots/per_project_f1_gain.png"))